In [1]:
import os
from typing import List

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

/Users/shahad/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


True

In [2]:
def load_txt_files(folder_path):
    documents = []
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                documents.append(f.read())
    print(f"✅ Loaded {len(documents)} files")
    return documents
documents = load_txt_files("./docs")

✅ Loaded 29 files


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)

chunks = []
for doc in documents:
    chunks.extend(splitter.split_text(doc))

print(f"✅ Created {len(chunks)} chunks")


✅ Created 74 chunks


In [4]:
processed_chunks = []
for chunk_text in chunks:
    doc = Document(
        page_content=chunk_text,
        metadata={"original_content": chunk_text}
    )
    processed_chunks.append(doc)

print(f"✅ Created {len(processed_chunks)} documents")

✅ Created 74 documents


In [5]:
import os

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")
    
        
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [9]:
def generate_answer(query, db):
    retriever = db.as_retriever(search_kwargs={"k": 3})
    results = retriever.invoke(query)
    
    context = "\n\n".join([doc.page_content for doc in results])
    
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    
    prompt = f"""أنت مساعد إسعافات أولية متخصص.
اعتمد فقط على المعلومات الموجودة في الوثائق.

القواعد:
- أجب باللغة العربية فقط
- قدم اسعافات أولية واضحة ومباشرة
- استخدم خطوات مرقمة
- لا تضف معلومات من خارج البيانات
- اذا لم تذكر الفئة العمرية في البيانات قدم الاسعافات لجميع الفئات العمرية

الوثائق:
{context}

السؤال: {query}

الجواب:"""
    
    response = llm.invoke(prompt)
    return response.content

# Example query
answer = generate_answer("مكرونه", db)
print(answer)

عذرًا، لا أستطيع مساعدتك في هذا الطلب بناءً على الوثائق المتاحة. إذا كان لديك سؤال آخر يتعلق بالإسعافات الأولية، فلا تتردد في طرحه.
